# Smoke Test — Verify Pipeline Works
Quick 1-epoch test to verify the entire pipeline (manifest → dataset → training → upload) works before committing to a full 40-epoch run.

**Run this first** on a new Kaggle account to catch issues early.

In [ ]:
# Cell 1: Clone repo & install deps
import subprocess, sys, shutil
from pathlib import Path

REPO_DIR = Path("/kaggle/working/mfft_repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "https://github.com/MIHMahmudEli/ai-image-detection-research.git", str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))

subprocess.run(["pip", "install", "-q", "huggingface_hub", "open_clip_torch", "scipy", "python-dotenv", "tqdm"], check=False)
print("Ready.")

In [ ]:
# Cell 2: Verify setup
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    print(f"HF Token: {token[:8]}...")
except Exception as e:
    print(f"ERROR: {e}")

# Check attached datasets
from pathlib import Path
input_root = Path("/kaggle/input")
attached = [d.name for d in input_root.iterdir() if d.is_dir()] if input_root.exists() else []
print(f"Attached datasets ({len(attached)}): {attached}")

In [ ]:
# Cell 3: Run 1-epoch smoke test
import os
os.environ["MFFT_MODEL_VARIANT"] = "base"

script_path = str(REPO_DIR / "kaggle_train_resumable.py")
with open(script_path) as f:
    code = f.read()

# Override for smoke test: 1 epoch, patience 1
code = code.replace('MAX_EPOCHS = 40', 'MAX_EPOCHS = 1')
code = code.replace('PATIENCE = 8', 'PATIENCE = 1')

exec(compile(code, script_path, 'exec'))

In [ ]:
# Cell 4: PASS / FAIL summary
import torch
from pathlib import Path

separator = "=" * 60

try:
    # ── Detect status ──
    smoke_status = state.status if 'state' in dir() else "error"
    passed = smoke_status in ("completed", "early_stopped")

    # ── GPU info ──
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        gpu_str = f"{gpu_name} ({vram_gb:.1f} GB)"
    else:
        gpu_str = "CPU only"

    # ── Images resolved ──
    n_train = len(loader.train_data) if 'loader' in dir() else 0
    n_val   = len(loader.val_data)   if 'loader' in dir() else 0
    n_test  = len(loader.test_data)  if 'loader' in dir() else 0
    n_total = n_train + n_val + n_test

    # ── Training metrics ──
    best_f1    = state.best_val_macro_f1 * 100 if 'state' in dir() else 0
    best_auc   = state.best_val_auc            if 'state' in dir() else 0
    best_acc   = state.best_val_accuracy       if 'state' in dir() else 0
    best_epoch = state.best_model_epoch        if 'state' in dir() else 0
    run_id_str = run_id                        if 'run_id' in dir() else "unknown"
    elapsed_s  = elapsed                       if 'elapsed' in dir() else 0

    # ── Print ──
    print(separator)
    if passed:
        print("  SMOKE TEST PASSED")
    else:
        print("  SMOKE TEST FAILED")
    print(separator)
    print()

    print(f"  GPU:              {gpu_str}")
    print(f"  Run ID:           {run_id_str}")
    print(f"  Status:           {smoke_status}")
    print()

    print(f"  Images resolved:  {n_total} total ({n_train} train / {n_val} val / {n_test} test)")
    print()

    print(f"  Best epoch:       {best_epoch}")
    print(f"  Val accuracy:     {best_acc:.2f}%")
    print(f"  Val macro-F1:     {best_f1:.2f}%")
    print(f"  Val AUC-ROC:      {best_auc:.4f}")
    print(f"  Wall time:        {elapsed_s:.0f}s ({elapsed_s/60:.1f} min)")
    print()

    if passed:
        print(separator)
        print("  Ready for full training: run train_mfft_base.ipynb (40 epochs)")
        print(separator)
    else:
        print(separator)
        print("  Troubleshooting:")
        print("  1. Ensure all 11 datasets are attached in the Input panel")
        print("  2. Verify HF_TOKEN is set as a Kaggle Secret")
        print("  3. Enable Internet in notebook settings")
        print("  4. Check GPU is enabled (Settings → Accelerator → GPU)")
        print("  5. Inspect the error trace above for details")
        print(separator)

except Exception as e:
    print(separator)
    print("  SMOKE TEST FAILED")
    print(separator)
    print()
    print(f"  Exception: {type(e).__name__}: {e}")
    print()
    print("  Troubleshooting:")
    print("  1. Ensure all 11 datasets are attached in the Input panel")
    print("  2. Verify HF_TOKEN is set as a Kaggle Secret")
    print("  3. Enable Internet in notebook settings")
    print("  4. Check GPU is enabled (Settings → Accelerator → GPU)")
    print("  5. Inspect the error trace above for details")
    print(separator)